In [0]:
#%run ./translation_function ----- A décommmenter pour lancer les notebooks séparements

# build_nomenclatures

Construit les **dimensions de nomenclature** : une ligne par entité métier,
avec son code technique.

## Pourquoi elles sont nécessaires

Sans elles, la relation Power BI entre les faits et une table de traduction est
**plusieurs-à-plusieurs** : `dim_batches_specifications[id_good_variety]` n'est
pas unique (plusieurs batches par variété) et `dim_trad_variety[id_good_variety]`
non plus (une ligne par langue). Deux côtés « plusieurs ».

La nomenclature rétablit un schéma en étoile classique :

```
dim_batches_specifications  --*→1--  dim_variety  --1←*--  dim_trad_variety
                                   (1 ligne/variété)      (4 lignes/variété)
```

Chaque relation a un côté « 1 » identifié. Le modèle devient lisible, et le
`code` de la nomenclature sert de **colonne de tri** : l'ordre des libellés reste
stable d'une langue à l'autre.

Ces dimensions dépassent le seul besoin de traduction — une `dim_variety` avec
son code et ses attributs est utile en soi, pour ce projet comme pour les autres.

In [0]:
# Sources : les tables métier de la base PostgreSQL du front.
goods_species = spark.table(f"{source_catalog}.goods_species")
goods_varieties = spark.table(f"{source_catalog}.goods_varieties")
requirement_specifications = spark.table(f"{source_catalog}.requirement_specifications")
parameters_production_types = spark.table(f"{source_catalog}.parameters_production_types")
parameters_variables = spark.table(f"{source_catalog}.parameters_variables")
parameters_localizations = spark.table(f"{source_catalog}.parameters_localizations")
parameters_localization_groups = spark.table(f"{source_catalog}.parameters_localization_groups")
parameters_production_line_variables = spark.table(f"{source_catalog}.parameters_production_line_variables")
parameters_batch_note_categories = spark.table(f"{source_catalog}.parameters_batch_note_categories")

## Les nomenclatures

Toutes filtrées sur `deleted = false`, comme le reste du pipeline.

Le `code` est conservé : c'est le libellé technique, non traduit, qui servira de
colonne de tri dans Power BI et de repère pour le debug.

In [0]:
# goods_species -> dim_specy
dim_specy = (
    goods_species
    .filter(F.col("deleted") == False)
    .select(
        F.col("id_good_specy"),
        F.col("code").alias("specy_code"),
    )
)

publish_dim(dim_specy, "dim_specy", ["id_good_specy"], translations_schema,
            is_translation=False)

In [0]:
# goods_varieties -> dim_variety
# La variété porte sa propre espèce (colonne specy) : on la garde, elle permet
# une hiérarchie espèce > variété dans le modèle.
dim_variety = (
    goods_varieties
    .filter(F.col("deleted") == False)
    .select(
        F.col("id_good_variety"),
        F.col("code").alias("variety_code"),
        F.col("specy").alias("id_good_specy"),
    )
)

publish_dim(dim_variety, "dim_variety", ["id_good_variety"], translations_schema,
            is_translation=False)

In [0]:
# parameters_production_types -> dim_production_type
dim_production_type = (
    parameters_production_types
    .filter(F.col("deleted") == False)
    .select(
        F.col("id_parameter_production_type"),
        F.col("code").alias("production_type_code"),
    )
)

publish_dim(dim_production_type, "dim_production_type",
            ["id_parameter_production_type"], translations_schema,
            is_translation=False)

In [0]:
# parameters_variables -> dim_variable
dim_variable = (
    parameters_variables
    .filter(F.col("deleted") == False)
    .select(
        F.col("id_parameter_variable"),
        F.col("code").alias("variable_code"),
    )
)

publish_dim(dim_variable, "dim_variable", ["id_parameter_variable"],
            translations_schema, is_translation=False)

## `requirement_specifications` — sans table de traduction

Cette nomenclature est construite pour compléter le modèle (relation propre,
colonne de tri), mais **il n'existe pas de `requirement_specifications_translations`
en base**. Son libellé restera donc dans la langue de saisie.

À confirmer avec le PO : oubli du front, ou champ volontairement non traduit ?

In [0]:
# requirement_specifications -> dim_requirement_specification
dim_requirement_specification = (
    requirement_specifications
    .filter(F.col("deleted") == False)
    .select(
        F.col("id_requirement_specification"),
        F.col("name").alias("requirement_specification_name"),
    )
)

publish_dim(dim_requirement_specification, "dim_requirement_specification",
            ["id_requirement_specification"], translations_schema,
            is_translation=False)

## `parameters_localizations`

La colonne `group` porte le rattachement au groupe de localisation. `group` est un
mot réservé SQL : il faut le protéger par des accents graves dans Spark, d'où le
`F.col("`group`")`.

In [0]:
dim_localization = (
    parameters_localizations
    .filter(F.col("deleted") == False)
    .select(
        F.col("id_parameter_localization"),
        F.col("code").alias("localization_code"),
        F.col("`group`").alias("id_parameter_localization_group"),
        F.col("sequence").alias("localization_sequence"),
    )
)

publish_dim(dim_localization, "dim_localization", ["id_parameter_localization"],
            translations_schema, is_translation=False)

## `parameters_localization_groups` — attention, clé asymétrique

La nomenclature est identifiée par `id_parameter_localization_group` **seul**,
alors que sa table de traduction a une clé **double** :
(`id_parameter_localization_group`, `production_line`). Un même groupe porte donc
plusieurs libellés, un par ligne de production.

> ⚠️ **Conséquence à traiter côté Power BI.** Après filtrage RLS sur la langue,
> `dim_trad_localization_group` conserve encore **une ligne par ligne de
> production** pour un même groupe. La garantie « exactement une ligne par clé »
> ne tient donc pas ici, et un visuel dupliquerait les lignes.
>
> Il faut soit filtrer aussi sur la ligne de production (clé composite, à porter
> par une colonne de substitution puisque Power BI ne gère pas les relations sur
> plusieurs colonnes), soit n'utiliser que
> `parameters_localizations_translations`, dont la clé est simple et suffit pour
> la page Pareto - Zoom Location.
>
> À arbitrer avec le PO : le libellé du groupe varie-t-il réellement d'une ligne
> de production à l'autre, ou est-ce une possibilité jamais utilisée ?

In [0]:
dim_localization_group = (
    parameters_localization_groups
    .filter(F.col("deleted") == False)
    .select(
        F.col("id_parameter_localization_group"),
        F.col("code").alias("localization_group_code"),
        F.col("sequence").alias("localization_group_sequence"),
    )
)

publish_dim(dim_localization_group, "dim_localization_group",
            ["id_parameter_localization_group"], translations_schema,
            is_translation=False)

## `parameters_production_line_variables`

Cette table n'a **pas de colonne `code`** : le libellé n'existe que dans la table
de traduction. On expose donc les clés de rattachement (`variable` vers
`parameters_variables`, `production_line`) et l'ordre d'affichage.

`displayed` est conservée : elle permettra de masquer dans le rapport les
paramètres que le front ne veut pas exposer.

In [0]:
dim_production_line_variable = (
    parameters_production_line_variables
    .filter(F.col("deleted") == False)
    .select(
        F.col("id_parameter_production_line_variable"),
        F.col("variable").alias("id_parameter_variable"),
        F.col("production_line").alias("id_plant_production_line"),
        F.col("sequence").alias("production_line_variable_sequence"),
        F.col("displayed"),
    )
)

publish_dim(dim_production_line_variable, "dim_production_line_variable",
            ["id_parameter_production_line_variable"], translations_schema,
            is_translation=False)

## `parameters_batch_note_categories`

Cette table résout proprement le découpage des notes de production :
`category_class` porte le type (`location`, `event`, `detail`, `impact`), là où
jusqu'ici je le déduisais du préfixe de la clé par découpage de chaîne.

Elle apporte aussi deux choses utiles :
- `category_label` : le libellé non traduit, meilleur repli de dernier recours
  que la clé technique ;
- `category_order` : l'ordre d'affichage voulu par le métier, qui servira de
  colonne de tri et restera stable entre les langues.

On produit **4 nomenclatures**, une par type, en miroir des 4 tables de
traduction — c'est ce qui donne 4 relations actives dans Power BI.

In [0]:
batch_note_categories_actives = (
    parameters_batch_note_categories
    .filter(F.col("deleted") == False)
    .select(
        F.col("id_batch_note_category").alias("batch_note_category"),
        F.col("category_class"),
        F.col("category_label"),
        F.col("category_order"),
    )
)

if verbose_mode == 'debug':
    print("Répartition par category_class :")
    display(batch_note_categories_actives.groupBy("category_class").count())

In [0]:
for category_type in ["location", "event", "detail", "impact"]:
    dim_type = (
        batch_note_categories_actives
        .filter(F.col("category_class") == category_type)
        .drop("category_class")
    )

    process_name = f"dim_batch_note_{category_type}"
    globals()[process_name] = dim_type
    publish_dim(dim_type, process_name, ["batch_note_category"],
                translations_schema, is_translation=False)

## Périmètre couvert

Les 9 nomenclatures du périmètre sont construites. Chaque table de traduction a
désormais son côté « 1 » : plus aucune relation plusieurs-à-plusieurs dans le
modèle, à l'exception de `dim_trad_localization_group` (voir l'avertissement sur
sa clé double plus haut).

Reste sans traduction possible : `requirement_specifications`, faute de table
`requirement_specifications_translations` en base.